In [ ]:
pip install -r requirements.txt

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
from cheb_ar.solvers.cheb_ar import *

In [ ]:
# ---------------- parameters (same as your snippet) ----------------
w_a = 25.338776456203686
w_b = 2 * w_a
kappa_b = 5 / 10.4
E_J = 37 * 2 * np.pi
phi_a, phi_b = 0.11, 0.204
epsilon_p = 0.1
g  = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_2 = 4 * g**2 / kappa_b
kappa_1 = 0.005 * kappa_2
n_a, n_b = 30, 11
dims = (n_a, n_b)
alpha_sq = 8.5
epsilon_d = 2 * alpha_sq * g2

N = n_a * n_b
T_block = 2 * jnp.pi / w_a
tsave = jnp.array([0.0, T_block])
method = dq.method.Tsit5(rtol=1e-9, atol=1e-10)
opts = dq.Options(assume_hermitian=False)

In [ ]:
def build_ats_hamiltonian_interaction(
    n_a=25,
    n_b=11,
    alpha_sq=8.5,
    w_a=25.338776456203686,
    kappa_b=5 / 10.4,
    E_J=37 * 2 * np.pi,
    phi_a=0.11,
    phi_b=0.204,
    epsilon_p=0.1,
    n_periods=1,
):
    """Build the driven full-ATS Lindbladian in the interaction frame.

    Returns
    -------
    H_I : time-dependent dynamiqs operator
        Driving part of the Hamiltonian.
    jump_ops : list
        Time-dependent Lindblad jump operators ``[sqrt(kappa_1) a, sqrt(kappa_b) b]``.
    T_block : float
        Duration of one Floquet block, ``n_periods * 2*pi / w_a``.
    params : dict
        Derived quantities (``g``, ``g2``, ``kappa_1``, ``kappa_2``,
        ``epsilon_d``, ``w_b``, ``T_drive``) for reference.
    """
    w_b = 2 * w_a

    # couplings derived from the pump
    g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
    g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b

    kappa_2 = 4 * g**2 / kappa_b
    kappa_1 = 0.005 * kappa_2

    epsilon_d = 2 * alpha_sq * g2

    # Operators
    a = dq.destroy(n_a)
    b = dq.destroy(n_b)
    Ia = dq.eye(n_a)
    Ib = dq.eye(n_b)

    a_tot = dq.tensor(a, Ib)
    b_tot = dq.tensor(Ia, b)

    phi_a_tot = phi_a*(a_tot + dq.dag(a_tot))
    phi_b_tot = phi_b*(b_tot + dq.dag(b_tot))

    non_linear_op = dq.sinm(phi_a_tot + phi_b_tot) - phi_a_tot - phi_b_tot

    H_s = (w_a * dq.dag(a_tot) @ a_tot + w_b * dq.dag(b_tot) @ b_tot
           - 2 * E_J * jnp.sin(epsilon_p) * non_linear_op)

    Hs_mat = H_s.to_numpy()
    Hs_mat = 0.5 * (Hs_mat + Hs_mat.conj().T)          # symmetrize numerically
    lam, V = np.linalg.eigh(Hs_mat)                    # H_s = V diag(lam) V^dag
    Vd = V.conj().T

    a_bar = jnp.array(Vd @ a_tot.to_numpy() @ V)        # fixed, built once
    b_bar = jnp.array(Vd @ b_tot.to_numpy() @ V)
    Delta = jnp.array(lam[:, None] - lam[None, :])      # fixed, built once

    def a_tilde_fn(t):
        return dq.asqarray(a_bar * jnp.exp(1j*Delta*t), dims=(n_a, n_b))

    def b_tilde_fn(t):
        return dq.asqarray(b_bar * jnp.exp(1j*Delta*t), dims=(n_a, n_b))

    a_tilde = dq.timecallable(a_tilde_fn)
    b_tilde = dq.timecallable(b_tilde_fn)

    def H_drive_I_fn(t):
        bt = b_tilde_fn(t)
        return epsilon_d * jnp.cos(w_b * t) * (bt + dq.dag(bt))

    H_I = dq.timecallable(H_drive_I_fn)
    jump_ops_I = [jnp.sqrt(kappa_1) * a_tilde, jnp.sqrt(kappa_b) * b_tilde]


    a_dag_a_bar = a_bar.conj().T @ a_bar
    b_dag_b_bar = b_bar.conj().T @ b_bar

    def a_dag_a_tilde_fn(t):
        return dq.asqarray(a_dag_a_bar * jnp.exp(1j * Delta * t), dims=(n_a, n_b))

    def b_dag_b_tilde_fn(t):
        return dq.asqarray(b_dag_b_bar * jnp.exp(1j * Delta * t), dims=(n_a, n_b))

    a_dag_a_tilde = dq.timecallable(a_dag_a_tilde_fn)
    b_dag_b_tilde = dq.timecallable(b_dag_b_tilde_fn)

    jump_ops_LdL_I = [kappa_1 * a_dag_a_tilde, kappa_b * b_dag_b_tilde]

    T_drive = 2 * jnp.pi / w_a
    T_block = n_periods * T_drive

    
    output_phase = jnp.exp(-1j * jnp.array(lam) * T_block)   # the missing piece

    params = {
        "g": g,
        "g2": g2,
        "kappa_1": kappa_1,
        "kappa_2": kappa_2,
        "kappa_b": kappa_b,
        "epsilon_d": epsilon_d,
        "w_a": w_a,
        "w_b": w_b,
        "T_drive": T_drive,
    }
    return H_I, jump_ops_I, jump_ops_LdL_I, output_phase, V, T_block, params

In [ ]:
H_I, jump_ops_I, jump_ops_LdL_I, output_phase, V, T_block, params  = build_ats_hamiltonian_interaction(n_a = n_a, n_b = n_b, alpha_sq = alpha_sq, epsilon_p = epsilon_p)
solver = ChebAr(H_I, jump_ops_I, T_block, jump_ops_LdL=jump_ops_LdL_I, dims=dims, output_phase=output_phase)
m_arnoldi_0 = 120
x0 = solver.make_x0(seed=0)
_, _, ritz_vals = solver.first_estimation(x0, m_arnoldi=m_arnoldi_0)
margin = 1e-2
solver.setup_chebyshev(ritz_vals, margin=margin)
warm_start = False
m_arnoldi = 120
Q, H, mu_list = solver.arnoldi_hessenberg(x0, solver.chebyshev_filter, m_arnoldi, warm_start = warm_start)


In [ ]:
plt.scatter(range(m_arnoldi), np.abs(1-np.real(mu_list)))
plt.xlabel("Arnoldi iteration")
plt.ylabel(r"$1-|\mathrm{Re}(\mu)|$")
plt.yscale("log")

In [ ]:
solver.rate_from_mu(mu_list[-1])

In [ ]:
x_ritz, rho_ritz = solver.ritz_vector(Q, H, m_arnoldi, target=mu_list[-1])
theta = np.linspace(0,2*np.pi, 100)

fig ,ax = plt.subplots()
dq.plot.wigner(dq.ptrace(rho_ritz,0), ax = ax)
ax.plot(np.sqrt(alpha_sq/2) * np.cos(theta), np.sqrt(alpha_sq/2) * np.sin(theta), color='r', 
        linewidth = 1, linestyle = '--')
plt.show()

In [ ]:
res = solver.residual_check(x_ritz)
res